# **Main functions**

In [2]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display

def order_points(pts):
    rect = np.zeros((4, 2), dtype="float32")
    
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]

    d = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(d)]
    rect[3] = pts[np.argmax(d)]

    return rect

def four_point_transform(image, pts):
    rect = order_points(pts)
    (tl, tr, br, bl) = rect

    widthA = np.linalg.norm(br-bl)
    widthB = np.linalg.norm(tr-tl)
    maxWidth = int(max(widthA, widthB))

    heightA = np.linalg.norm(tr-br)
    heightB = np.linalg.norm(tl-bl)
    maxHeight = int(max(heightA, heightB))

    margin = 20
    
    dst = np.array([
        [margin, margin],
        [margin + maxWidth - 1, margin],
        [margin + maxWidth - 1, margin + maxHeight - 1],
        [margin, margin + maxHeight-1]
    ], dtype="float32")

    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, M, (maxWidth + 2*margin, maxHeight + 2*margin))
    return warped
    
def detect_walls(img):
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    #thresh = cv2.adaptiveThreshold(img_gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, 81, 31)
    _, thresh = cv2.threshold(img_gray, 60, 255, cv2.THRESH_BINARY_INV)
    
    analysis = cv2.connectedComponentsWithStats(thresh, 4, cv2.CV_32S)
    (totalLabels, label_ids, values, centroid) = analysis
    
    walls = np.zeros(thresh.shape, dtype="uint8")
    min_area = 6000
    
    for i in range(1, totalLabels):
        area = values[i, cv2.CC_STAT_AREA]  

        if (area > min_area) :
            componentMask = (label_ids == i).astype("uint8") * 255
            walls = cv2.bitwise_or(walls, componentMask)
            
    return walls
    
def add_safety_margin(grid, margin=3):
    wall_mask = (grid == 1).astype(np.uint8)

    size = 2*margin + 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))
    dilated = cv2.dilate(wall_mask, kernel)

    grid[(dilated == 1) & (grid == 0)] = 4
    return grid

def to_grid_walls(walls, margin, rows, cols):
    grid = overlay = np.zeros((rows, cols), dtype=np.uint8)
    h, w = walls.shape
    
    for r in range(rows):
        for c in range(cols):
            y_start = int(r*h/rows)
            y_end   = int((r + 1)*h/rows)
            x_start = int(c*w/cols)
            x_end   = int((c + 1)*w/cols)
            
            cell = walls[y_start:y_end, x_start:x_end]
            if np.mean(cell) > 100:
                grid[r, c] = 1 
            else:
                grid[r, c] = 0

    grid = add_safety_margin(grid, margin=margin)
    
    return grid

def to_grid_pos(grid, thymio_coord, finish_coord, rows, cols, h, w):
    if thymio_coord != None:
        x_thymio, y_thymio = thymio_coord[0]
        
        c = int(x_thymio*cols/w)
        r = int(y_thymio*rows/h)
        c_end = int(c + 5*np.cos(thymio_coord[1]))
        r_end = int(r + 5*np.sin(thymio_coord[1]))
        
        grid[r, c] = 2
        grid = cv2.circle(grid, (c, r), 5, 2, 1)
        grid = cv2.line(grid, (c, r), (c_end, r_end), 2, 1)

    if finish_coord != None:
        x_finish, y_finish = finish_coord
        r = int(y_finish*rows/h)
        c = int(x_finish*cols/w)
        grid[r, c] = 3
    
    return grid

def draw_grid(grid):
    rows, cols = grid.shape
    img = np.full((rows, cols, 3), 255, dtype=np.uint8)   # white background (BGR)

    img[grid == 1] = (0, 0, 0)
    img[grid == 2] = (0, 200, 0)
    img[grid == 3] = (0, 0, 255)
    img[grid == 4] = (180, 180, 180)

    return img
    
def find_light(mask, min_area=20):
    contour, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contour:
        return None

    largest = max(contour, key=cv2.contourArea)
    
    if cv2.contourArea(largest) < min_area:
        return None
    
    M = cv2.moments(largest)

    if M['m00'] == 0:
        return None
    
    cx = int(M['m10']/M['m00'])
    cy = int(M['m01']/M['m00'])

    return cx, cy

def detect_finish(img):
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    lower1 = np.array([0, 120, 70])
    upper1 = np.array([10, 255, 255])

    lower2 = np.array([170, 120, 70])
    upper2 = np.array([179, 255, 255])

    mask1 = cv2.inRange(img_hsv, lower1, upper1)
    mask2 = cv2.inRange(img_hsv, lower2, upper2)
    mask = cv2.bitwise_or(mask1, mask2)

    kernel = np.ones((5, 5), np.uint8)

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    finish_coord = find_light(mask)

    if finish_coord == None:
        return None

    return finish_coord

def detect_thymio(img):
    hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    lower_g = np.array([35, 80, 80])
    upper_g = np.array([85, 255, 255])
    lower_b = np.array([90, 80, 80])
    upper_b = np.array([120, 255, 255])
    
    mask_g = cv2.inRange(hsv, lower_g, upper_g)
    mask_b = cv2.inRange(hsv, lower_b, upper_b)
    
    kernel = np.ones((5, 5), np.uint8)
    
    mask_g = cv2.morphologyEx(mask_g, cv2.MORPH_OPEN, kernel)
    mask_b = cv2.morphologyEx(mask_b, cv2.MORPH_OPEN, kernel)

    light_g = find_light(mask_g)
    light_b = find_light(mask_b)
    
    if light_g == None or light_b == None:
        return None

    cx_g, cy_g = light_g
    cx_b, cy_b = light_b

    center_g = [cx_g, cy_g]
    center_b = [cx_b, cy_b]
    
    theta = np.arctan2(cy_g - cy_b, cx_g - cx_b)
    center = [(cx_g + cx_b)//2, (cy_g + cy_b)//2]
    
    return center, theta

def draw_direction(img, coordinates):
    if coordinates == None :
        return img, None

    center, theta = coordinates
    
    img_full = cv2.circle(img, center, 5, (0, 0, 255), -1)

    x_end = int(center[0] + 100*np.cos(theta))
    y_end = int(center[1] + 100*np.sin(theta))

    img_full = cv2.arrowedLine(img_full, center, (x_end, y_end), (100, 100, 0), 5)

    sz = img.shape
    h, w = sz[0], sz[1]
    
    overlay = np.zeros((h, w, 4), dtype=np.uint8)
    cv2.circle(overlay, center, 5, (0, 0, 255, 255), -1)
    cv2.arrowedLine(overlay, center, (x_end, y_end), (100, 100, 0, 255), 5)
    
    return img_full, overlay

def draw_finish(img, coordinates):
    if coordinates == None:
        return None
    
    sz = img.shape
    h, w = sz[0], sz[1]
    
    overlay = np.zeros((h, w, 4), dtype=np.uint8)
    cv2.circle(overlay, coordinates, 5, (0, 0, 255, 255), -1)

    return overlay

def merge(base_bgr, overlay_bgra):
    overlay_bgra = cv2.resize(overlay_bgra, (base_bgr.shape[1], base_bgr.shape[0]), interpolation=cv2.INTER_NEAREST)
    alpha = overlay_bgra[:, :, 3:4] / 255.0 
    out = base_bgr * (1 - alpha) + overlay_bgra[:, :, :3] * alpha
    return out.astype(np.uint8)

def capture_obstacles(sec, cap, rows, cols):
    for i in range(sec*10):
        ok, frame = cap.read()
        _, jpeg = cv2.imencode('.jpg', frame)
        ecran.value = jpeg.tobytes()
        
    ok, frame = cap.read()
    
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gauss_frame = cv2.GaussianBlur(gray_frame, (5, 5), 0)
    filtered_frame = cv2.Canny(gauss_frame, 50, 150)
    contours_frame, hierarchy = cv2.findContours(filtered_frame, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours_frame:
        largest = max(contours_frame, key=cv2.contourArea)
        epsilon = 0.02 * cv2.arcLength(largest, True)
        approx = cv2.approxPolyDP(largest, epsilon, True)

        if len(approx) == 4:
            pts = approx.reshape(4, 2)
    
            warped = four_point_transform(frame, pts.astype("float32"))
            walls = detect_walls(warped)

            h, w = walls.shape
            
            grid = to_grid_walls(walls, 15, rows, cols)
    
    return grid, h, w

**A*** **test**

In [18]:
import heapq
import time

def pos_to_cell(pt, rows, cols, h, w):
    """Pixel (x, y) in the warped image -> grid cell (row, col)."""
    x, y = pt
    r = min(max(int(y * rows / h), 0), rows - 1)
    c = min(max(int(x * cols / w), 0), cols - 1)
    return r, c

def cell_to_pos(cell, rows, cols, h, w):
    """Grid cell (row, col) -> pixel (x, y) at the center of the cell."""
    r, c = cell
    return int((c + 0.5) * w / cols), int((r + 0.5) * h / rows)


def astar(grid, start, goal, block_margin=True, margin_cost=5.0):
    rows, cols = grid.shape
    SQRT2 = 2 ** 0.5
    moves = [(-1, 0, 1), (1, 0, 1), (0, -1, 1), (0, 1, 1),
             (-1, -1, SQRT2), (-1, 1, SQRT2), (1, -1, SQRT2), (1, 1, SQRT2)]

    def blocked(r, c):
        v = grid[r, c]
        return v == 1 or (block_margin and v == 4)

    if blocked(*start) or blocked(*goal):
        return None

    def heuristic(cell):
        dr, dc = abs(cell[0] - goal[0]), abs(cell[1] - goal[1])
        return (dr + dc) + (SQRT2 - 2) * min(dr, dc)

    open_heap = [(heuristic(start), 0.0, start)]
    came_from = {}
    best_g = {start: 0.0}
    closed = set()

    while open_heap:
        _, g, current = heapq.heappop(open_heap)
        if current in closed:
            continue
        if current == goal:
            path = [current]
            while current in came_from:
                current = came_from[current]
                path.append(current)
            return path[::-1]
        closed.add(current)

        r, c = current
        for dr, dc, step in moves:
            nr, nc = r + dr, c + dc
            if not (0 <= nr < rows and 0 <= nc < cols) or blocked(nr, nc):
                continue
            if dr != 0 and dc != 0 and (blocked(r + dr, c) or blocked(r, c + dc)):
                continue
            cost = step * (1 + (margin_cost if grid[nr, nc] == 4 else 0))
            ng = g + cost
            if ng < best_g.get((nr, nc), float("inf")):
                best_g[(nr, nc)] = ng
                came_from[(nr, nc)] = current
                heapq.heappush(open_heap, (ng + heuristic((nr, nc)), ng, (nr, nc)))
    return None


def animate_path(base_img, grid, path, screen, theta0=None,
                 delay=0.05, stride=1, scale=0.5, loop=True, pause_end=1.0):
    rows, cols = grid.shape
    h, w = base_img.shape[:2]
    pts = [cell_to_pos(cell, rows, cols, h, w) for cell in path]

    left_bg = base_img.copy()
    cv2.polylines(left_bg, [np.array(pts, dtype=np.int32)], False, (255, 150, 0), 2)
    cv2.circle(left_bg, pts[-1], 8, (0, 0, 255), -1)

    right_bg = draw_grid(grid)
    for (r, c) in path:
        right_bg[r, c] = (255, 150, 0)
    right_bg[path[-1]] = (0, 0, 255)
    right_bg = cv2.resize(right_bg, (w, h), interpolation=cv2.INTER_NEAREST)

    indices = list(range(0, len(pts), stride))
    if indices[-1] != len(pts) - 1:
        indices.append(len(pts) - 1)

    try:
        while True:
            theta = theta0 if theta0 is not None else 0.0     # restart from the real heading

            for i in indices:
                left, right = left_bg.copy(), right_bg.copy()

                if i > 0:
                    cv2.polylines(left, [np.array(pts[:i + 1], dtype=np.int32)], False, (0, 200, 0), 3)

                j = min(i + 5, len(pts) - 1)
                dx, dy = pts[j][0] - pts[i][0], pts[j][1] - pts[i][1]
                if i == 0 and theta0 is not None:
                    theta = theta0
                elif dx or dy:
                    theta = np.arctan2(dy, dx)

                x, y = pts[i]
                tip = (int(x + 30 * np.cos(theta)), int(y + 30 * np.sin(theta)))
                for panel in (left, right):
                    cv2.circle(panel, (x, y), 12, (0, 165, 255), -1)
                    cv2.arrowedLine(panel, (x, y), tip, (0, 0, 0), 3)

                both = np.hstack([left, right])
                both = cv2.resize(both, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
                _, jpeg = cv2.imencode('.jpg', both)
                screen.value = jpeg.tobytes()
                time.sleep(delay)

            time.sleep(pause_end)        # short pause at the finish before restarting
            if not loop:
                break
    except KeyboardInterrupt:
        pass                             # the Interrupt button ends the loop cleanly

# **Main loop**

In [23]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from time import sleep

In [40]:
cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

ecran = widgets.Image(format="jpeg")
display(ecran)

rows = 160
cols = 200

obstacles_grid, h, w = capture_obstacles(5, cap, rows, cols)
    
while True:
    ok, frame = cap.read()
    if not ok:
        break
        
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gauss_frame = cv2.GaussianBlur(gray_frame, (5, 5), 0)
    filtered_frame = cv2.Canny(gauss_frame, 50, 150)
    contours_frame, hierarchy = cv2.findContours(filtered_frame, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours_frame:
        largest = max(contours_frame, key=cv2.contourArea)
        epsilon = 0.02 * cv2.arcLength(largest, True)
        approx = cv2.approxPolyDP(largest, epsilon, True)

        if len(approx) == 4:
            pts = approx.reshape(4, 2)
    
            warped = four_point_transform(frame, pts.astype("float32"))
            
            finish_coord = detect_finish(warped)
            thymio_coord = detect_thymio(warped)
            
            grid_full = to_grid_pos(obstacles_grid.copy(), thymio_coord, finish_coord, rows, cols, h, w)
            grid_img = draw_grid(grid_full)
            grid_img = cv2.resize(grid_img, (w, h), interpolation=cv2.INTER_NEAREST)
            
            _, jpeg = cv2.imencode('.jpg', grid_img)
            ecran.value = jpeg.tobytes()
            
            continue
            
    _, jpeg = cv2.imencode('.jpg', frame)
    ecran.value = jpeg.tobytes()

Image(value=b'', format='jpeg')

i
i


KeyboardInterrupt: 

In [3]:
cap.release()

# **Capture 1 image**

In [13]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display

cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

ecran = widgets.Image(format="jpeg")
display(ecran)

for _ in range(50):
    ok, frame = cap.read()
    _, jpeg = cv2.imencode('.jpg', frame)
    ecran.value = jpeg.tobytes()
    
ok, frame = cap.read()
if ok:
    cv2.imwrite("photo_test_environment.jpg", frame) # nom à changer

cap.release()



Image(value=b'', format='jpeg')

In [138]:
cap.release()

In [4]:
cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

ecran = widgets.Image(format="jpeg")
display(ecran)

obstacles_grid = capture_obstacles(5, cap)
cap.release()

print(obstacles_grid)

Image(value=b'', format='jpeg')

KeyboardInterrupt: 

In [2]:
for i in range(5):
    cap = cv2.VideoCapture(i, cv2.CAP_DSHOW)
    ok, frame = cap.read()
    print(i, "OK" if ok else "nothing", frame.shape if ok else "")
    cap.release()

0 OK (480, 640, 3)
1 OK (480, 640, 3)
2 nothing 
3 nothing 
4 nothing 


In [22]:
def repeat_until_yes():
    while True:
        # --- placeholder: your code goes here ---

        answer = input("Are you done? (yes/no): ").strip().lower()
        if answer in ("yes", "y"):
            break

repeat_until_yes()

Are you done? (yes/no):  n
Are you done? (yes/no):  n
Are you done? (yes/no):  n
Are you done? (yes/no):  n
Are you done? (yes/no):  y


# **Walls without thymio testing (static)**

In [96]:
img = cv2.imread('photo_thymio.jpg')
ecran = widgets.Image(format="jpeg")
display(ecran)

img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

_, img_gray_thresh = cv2.threshold(img_gray, 40, 255, cv2.THRESH_BINARY)
_, img_hsv_thresh = cv2.threshold(img_hsv[:, :, 2], 40, 255, cv2.THRESH_BINARY)

'''
print(center, theta)

img = cv2.circle(img, center, 5, (0, 0, 255), -1)

x_end = int(center[0] + 200*np.cos(theta))
y_end = int(center[1] + 200*np.sin(theta))

img = cv2.arrowedLine(img, center, (x_end, y_end), (100, 100, 0), 5)

row1 = np.hstack([img_gray, img_hsv[:, :, 2]])
row2 = np.hstack([img_gray_thresh, img_hsv_thresh])
full = np.vstack([row1, row2])
'''

filtered_frame = cv2.Canny(img_gray, 50, 150)
contours_frame, hierarchy = cv2.findContours(filtered_frame, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

if contours_frame:
    largest = max(contours_frame, key=cv2.contourArea)
    epsilon = 0.02 * cv2.arcLength(largest, True)
    approx = cv2.approxPolyDP(largest, epsilon, True)

    if len(approx) == 4:
        pts = approx.reshape(4, 2)

        warped = four_point_transform(img, pts.astype("float32"))

        img_gray = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
        img_hsv = cv2.cvtColor(warped, cv2.COLOR_BGR2HSV)
        walls = detect_walls(warped)
        #_, walls = cv2.threshold(img_gray, 40, 255, cv2.THRESH_BINARY_INV)
        #walls_bgr = cv2.cvtColor(walls, cv2.COLOR_GRAY2BGR)
        walls_blurred = cv2.GaussianBlur(walls, (3, 3), 0)
        
        analysis = cv2.connectedComponentsWithStats(walls_blurred, 4, cv2.CV_32S)
        (totalLabels, label_ids, values, centroid) = analysis
        
        output = np.zeros(walls.shape, dtype="uint8")
        min_area = 3000
        
        for i in range(1, totalLabels):
            area = values[i, cv2.CC_STAT_AREA]  
    
            if (area > min_area) :
                componentMask = (label_ids == i).astype("uint8") * 255
                output = cv2.bitwise_or(output, componentMask)
        
        h1 = np.hstack([img_gray, output])
        
        _, jpeg = cv2.imencode('.jpg', h1)
        ecran.value = jpeg.tobytes()

Image(value=b'', format='jpeg')

# **Object testing (dynamic)**

In [5]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display

cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

ecran = widgets.Image(format="jpeg")
display(ecran)

ecran_grid = widgets.Image(format="jpeg")
display(ecran_grid)

while True:
    ok, frame = cap.read()
    if not ok:
        break
        
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gauss_frame = cv2.GaussianBlur(gray_frame, (5, 5), 0)
    filtered_frame = cv2.Canny(gauss_frame, 50, 150)
    contours_frame, hierarchy = cv2.findContours(filtered_frame, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours_frame:
        largest = max(contours_frame, key=cv2.contourArea)
        epsilon = 0.02 * cv2.arcLength(largest, True)
        approx = cv2.approxPolyDP(largest, epsilon, True)

        if len(approx) == 4:
            pts = approx.reshape(4, 2)
    
            warped = four_point_transform(frame, pts.astype("float32"))
            walls = detect_walls(warped)

            grid = to_grid(walls, coordinates, 100, 100)
            grid_img = draw_grid(grid, cell_px=50)
            
            h, w = walls.shape
            grid_img = cv2.resize(grid_img, (w, h), interpolation=cv2.INTER_NEAREST)
            both = np.hstack([grid_img, walls])

            _, jpeg = cv2.imencode('.jpg', both)
            ecran.value = jpeg.tobytes()
            
            continue
            
    _, jpeg = cv2.imencode('.jpg', frame)
    ecran.value = jpeg.tobytes()

Image(value=b'', format='jpeg')

Image(value=b'', format='jpeg')

KeyboardInterrupt: 

# **Object testing (static)**

In [10]:
img = cv2.imread('Photo_full_environment.jpg')
ecran = widgets.Image(format="jpeg")
display(ecran)

gray_frame = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gauss_frame = cv2.GaussianBlur(gray_frame, (5, 5), 0)
filtered_frame = cv2.Canny(gauss_frame, 50, 150)
contours_frame, hierarchy = cv2.findContours(filtered_frame, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

if contours_frame:
    largest = max(contours_frame, key=cv2.contourArea)
    epsilon = 0.02 * cv2.arcLength(largest, True)
    approx = cv2.approxPolyDP(largest, epsilon, True)

    if len(approx) == 4:
        pts = approx.reshape(4, 2)

        warped = four_point_transform(img, pts.astype("float32"))
        walls = detect_walls(warped)

        coordinates = None
        finish_coord = detect_finish(warped, 
        
        grid = to_grid(walls, coordinates, 100, 100)
        grid_img = draw_grid(grid, cell_px=50)
        
        h, w = walls.shape
        grid_img = cv2.resize(grid_img, (w, h), interpolation=cv2.INTER_NEAREST)
        both = np.vstack([walls, grid_img])

        _, jpeg = cv2.imencode('.jpg', both)
        ecran.value = jpeg.tobytes()

Image(value=b'', format='jpeg')

# **Thymio and objects testing (static)**

In [50]:
img = cv2.imread('photo_objects_thymio.jpg')
ecran = widgets.Image(format="jpeg")
display(ecran)

gray_frame = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gauss_frame = cv2.GaussianBlur(gray_frame, (5, 5), 0)
filtered_frame = cv2.Canny(gauss_frame, 50, 150)
contours_frame, hierarchy = cv2.findContours(filtered_frame, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

if contours_frame:
    largest = max(contours_frame, key=cv2.contourArea)
    epsilon = 0.02 * cv2.arcLength(largest, True)
    approx = cv2.approxPolyDP(largest, epsilon, True)

    if len(approx) == 4:
        pts = approx.reshape(4, 2)

        warped = four_point_transform(img, pts.astype("float32"))
        walls = detect_walls(warped)

        coordinates = detect_thymio(warped)
        
        h, w = walls.shape

        grid = to_grid(walls, 15, h//3, w//3)
        grid_img = draw_grid(grid, cell_px=50)
        
        warped, directions = draw_direction(warped, coordinates)
        
        grid_img = cv2.resize(grid_img, (w, h), interpolation=cv2.INTER_NEAREST)
        grid_bgr = cv2.cvtColor(grid_img, cv2.COLOR_GRAY2BGR)
        grid_with_dir = merge(grid_bgr, directions)
        
        walls_bgr = cv2.cvtColor(walls, cv2.COLOR_GRAY2BGR)
        both = np.vstack([warped, walls_bgr, grid_with_dir])

        _, jpeg = cv2.imencode('.jpg', both)
        ecran.value = jpeg.tobytes()

Image(value=b'', format='jpeg')

# **Find finish (static)**

In [16]:
img = cv2.imread('photo_test_environment.jpg')
ecran = widgets.Image(format="jpeg")
display(ecran)

gray_frame = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gauss_frame = cv2.GaussianBlur(gray_frame, (5, 5), 0)
filtered_frame = cv2.Canny(gauss_frame, 50, 150)
contours_frame, hierarchy = cv2.findContours(filtered_frame, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

if contours_frame:
    largest = max(contours_frame, key=cv2.contourArea)
    epsilon = 0.02 * cv2.arcLength(largest, True)
    approx = cv2.approxPolyDP(largest, epsilon, True)

    if len(approx) == 4:
        pts = approx.reshape(4, 2)

        warped = four_point_transform(img, pts.astype("float32"))
        
        finish_coord = detect_finish(warped)
        thymio_coord = detect_thymio(warped)

        warped_w_dir, thymio_overlay = draw_direction(warped, thymio_coord)
        finish_overlay = draw_finish(warped, finish_coord)

        walls = detect_walls(warped)
        h, w = walls.shape
        
        rows = 160
        cols = 200
        
        grid = to_grid_walls(walls, 8, rows, cols)
        grid_full = to_grid_pos(grid, thymio_coord, finish_coord, rows, cols, h, w)
        grid_img = draw_grid(grid_full)
        grid_img = cv2.resize(grid_img, (w, h), interpolation=cv2.INTER_NEAREST)

        both = np.hstack([warped_w_dir, grid_img])
        
        _, jpeg = cv2.imencode('.jpg', both)
        ecran.value = jpeg.tobytes()

Image(value=b'', format='jpeg')

# **Finish (dynamic)**


In [46]:
cap = cv2.VideoCapture(1, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

ecran = widgets.Image(format="jpeg")
display(ecran)

ecran_grid = widgets.Image(format="jpeg")
display(ecran_grid)

while True:
    ok, frame = cap.read()
    if not ok:
        break
        
    gray_frame = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gauss_frame = cv2.GaussianBlur(gray_frame, (5, 5), 0)
    filtered_frame = cv2.Canny(gauss_frame, 50, 150)
    contours_frame, hierarchy = cv2.findContours(filtered_frame, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if contours_frame:
        largest = max(contours_frame, key=cv2.contourArea)
        epsilon = 0.02 * cv2.arcLength(largest, True)
        approx = cv2.approxPolyDP(largest, epsilon, True)
    
        if len(approx) == 4:
            pts = approx.reshape(4, 2)
    
            warped = four_point_transform(img, pts.astype("float32"))
            
            finish_coord = detect_finish(warped)
            thymio_coord = detect_thymio(warped)
    
            warped_w_dir, thymio_overlay = draw_direction(warped, thymio_coord)
            finish_overlay = draw_finish(warped, finish_coord)
    
            walls = detect_walls(warped)
            
            rows = 100
            cols = 100
            
            grid = to_grid_walls(walls, 3, rows, cols)
            grid_full = to_grid_pos(grid, thymio_coord, finish_coord, rows, cols)
            grid_img = draw_grid(grid_full)
            grid_img = cv2.resize(grid_img, (w, h), interpolation=cv2.INTER_NEAREST)
    
            both = np.hstack([warped_w_dir, grid_img])
            
            _, jpeg = cv2.imencode('.jpg', both)
            ecran.value = jpeg.tobytes()

            continue
            
    _, jpeg = cv2.imencode('.jpg', frame)
    ecran.value = jpeg.tobytes()

Image(value=b'', format='jpeg')

Image(value=b'', format='jpeg')

# **A*** **test**

In [19]:
img = cv2.imread('photo_test_environment.jpg')
ecran = widgets.Image(format="jpeg")
display(ecran)

gray_frame = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gauss_frame = cv2.GaussianBlur(gray_frame, (5, 5), 0)
filtered_frame = cv2.Canny(gauss_frame, 50, 150)
contours_frame, hierarchy = cv2.findContours(filtered_frame, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

if contours_frame:
    largest = max(contours_frame, key=cv2.contourArea)
    epsilon = 0.02 * cv2.arcLength(largest, True)
    approx = cv2.approxPolyDP(largest, epsilon, True)

    if len(approx) == 4:
        pts = approx.reshape(4, 2)

        warped = four_point_transform(img, pts.astype("float32"))
        warped_clean = warped.copy()                 # draw_direction paints on warped in place
        h, w = warped.shape[:2]
        
        finish_coord = detect_finish(warped)
        thymio_coord = detect_thymio(warped)

        warped_w_dir, thymio_overlay = draw_direction(warped, thymio_coord)
        finish_overlay = draw_finish(warped, finish_coord)

        walls = detect_walls(warped)
        
        rows = 200
        cols = 200
        
        grid = to_grid_walls(walls, 3, rows, cols)
        
        if thymio_coord is not None and finish_coord is not None:
            start = pos_to_cell(thymio_coord[0], rows, cols, h, w)
            goal = pos_to_cell(finish_coord, rows, cols, h, w)
        
            path = astar(grid, start, goal)
            print('bouh')
            if path is None:
                if grid[start] in (1, 4):
                    print("The Thymio's cell is inside a wall or margin zone")
                elif grid[goal] in (1, 4):
                    print("The finish cell is inside a wall or margin zone")
                else:
                    print("No path: the margin closes every passage")
            else:
                print(f"Path found: {len(path)} cells")
                animate_path(warped_clean, grid, path, ecran, theta0=thymio_coord[1])

        grid_full = to_grid_pos(grid, thymio_coord, finish_coord)
        grid_img = draw_grid(grid_full)
        grid_img = cv2.resize(grid_img, (w, h), interpolation=cv2.INTER_NEAREST)

        both = np.hstack([warped_w_dir, grid_img])
        
        _, jpeg = cv2.imencode('.jpg', both)
        ecran.value = jpeg.tobytes()

Image(value=b'', format='jpeg')

bouh
Path found: 121 cells


TypeError: to_grid_pos() missing 4 required positional arguments: 'rows', 'cols', 'h', and 'w'